In [0]:
spark.conf.set(
  "fs.azure.account.key.cloud1111.dfs.core.windows.net",
  "lPTzWBp5iVCNKhWA61BtDg7OBziO8OdZfR4o9GbCRAunak/PvRPjkGZXyMJx9WjBmg9YZFvuj/1r+ASt+/hHpA=="
)

In [0]:
dbutils.fs.ls("abfss://raw@cloud1111.dfs.core.windows.net/")

[FileInfo(path='abfss://raw@cloud1111.dfs.core.windows.net/bronze/', name='bronze/', size=0, modificationTime=1777552198000)]

In [0]:
dbutils.fs.mkdirs("abfss://raw@cloud1111.dfs.core.windows.net/bronze/sales/")

True

In [0]:
bronze_path = "abfss://raw@cloud1111.dfs.core.windows.net/bronze/sales/"

In [0]:
import requests
from pyspark.sql.functions import current_timestamp
from datetime import date

api_key = "bcf4a22702e1674e2913179b5ac928cb"

# ✅ List of cities
cities = ["London", "Paris", "New York", "Tokyo", "Delhi", "Bangalore"]

data_list = []

for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed for {city}: {response.text}")
        continue
    
    data = response.json()

    # ✅ Store city + raw JSON
    data_list.append((city, str(data)))

# ✅ Create DataFrame with all cities
df = spark.createDataFrame(data_list, ["city", "raw_json"])

# Add ingestion time
df = df.withColumn("ingestion_time", current_timestamp())

# Bronze path
today = date.today()
bronze_path = f"abfss://raw@cloud1111.dfs.core.windows.net/bronze/api/weather/ingestion_date={today}/"

# Write
df.write.format("delta").mode("append").save(bronze_path)

display(df)

city,raw_json,ingestion_time
London,"{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 13.15, 'feels_like': 12.17, 'temp_min': 12.27, 'temp_max': 15.06, 'pressure': 1016, 'humidity': 63, 'sea_level': 1016, 'grnd_level': 1012}, 'visibility': 10000, 'wind': {'speed': 2.06, 'deg': 90}, 'clouds': {'all': 20}, 'dt': 1778149586, 'sys': {'type': 2, 'id': 2075535, 'country': 'GB', 'sunrise': 1778127706, 'sunset': 1778182340}, 'timezone': 3600, 'id': 2643743, 'name': 'London', 'cod': 200}",2026-05-07T10:32:09.349Z
Paris,"{'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 15.13, 'feels_like': 14.19, 'temp_min': 14.09, 'temp_max': 16.7, 'pressure': 1016, 'humidity': 57, 'sea_level': 1016, 'grnd_level': 1006}, 'visibility': 10000, 'wind': {'speed': 2.06, 'deg': 20}, 'clouds': {'all': 75}, 'dt': 1778149434, 'sys': {'type': 2, 'id': 2012208, 'country': 'FR', 'sunrise': 1778127642, 'sunset': 1778181217}, 'timezone': 7200, 'id': 2988507, 'name': 'Paris', 'cod': 200}",2026-05-07T10:32:09.349Z
New York,"{'coord': {'lon': -74.006, 'lat': 40.7143}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 10.34, 'feels_like': 9.62, 'temp_min': 8.98, 'temp_max': 11.1, 'pressure': 1009, 'humidity': 84, 'sea_level': 1009, 'grnd_level': 1009}, 'visibility': 10000, 'wind': {'speed': 5.66, 'deg': 320, 'gust': 9.26}, 'clouds': {'all': 100}, 'dt': 1778149560, 'sys': {'type': 1, 'id': 4610, 'country': 'US', 'sunrise': 1778147242, 'sunset': 1778198265}, 'timezone': -14400, 'id': 5128581, 'name': 'New York', 'cod': 200}",2026-05-07T10:32:09.349Z
Tokyo,"{'coord': {'lon': 139.6917, 'lat': 35.6895}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 21.4, 'feels_like': 21.87, 'temp_min': 20.35, 'temp_max': 22.02, 'pressure': 1009, 'humidity': 87, 'sea_level': 1009, 'grnd_level': 1007}, 'visibility': 10000, 'wind': {'speed': 5.66, 'deg': 190}, 'clouds': {'all': 75}, 'dt': 1778149802, 'sys': {'type': 2, 'id': 268395, 'country': 'JP', 'sunrise': 1778096621, 'sunset': 1778146315}, 'timezone': 32400, 'id': 1850144, 'name': 'Tokyo', 'cod': 200}",2026-05-07T10:32:09.349Z
Delhi,"{'coord': {'lon': 77.2167, 'lat': 28.6667}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': 31.05, 'feels_like': 30.56, 'temp_min': 31.05, 'temp_max': 31.05, 'pressure': 1006, 'humidity': 37, 'sea_level': 1006, 'grnd_level': 978}, 'visibility': 6000, 'wind': {'speed': 7.72, 'deg': 110, 'gust': 12.86}, 'clouds': {'all': 0}, 'dt': 1778149588, 'sys': {'type': 1, 'id': 9165, 'country': 'IN', 'sunrise': 1778112350, 'sunset': 1778160573}, 'timezone': 19800, 'id': 1273294, 'name': 'Delhi', 'cod': 200}",2026-05-07T10:32:09.349Z
Bangalore,"{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03d'}], 'base': 'stations', 'main': {'temp': 32.36, 'feels_like': 33.2, 'temp_min': 30.9, 'temp_max': 33.26, 'pressure': 1005, 'humidity': 42, 'sea_level': 1005, 'grnd_level': 909}, 'visibility': 8000, 'wind': {'speed': 6.26, 'deg': 86, 'gust': 22.35}, 'clouds': {'all': 40}, 'dt': 1778149312, 'sys': {'type': 2, 'id': 2109756, 'country': 'IN', 'sunrise': 1778113601, 'sunset': 1778159136}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}",2026-05-07T10:32:09.349Z


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", IntegerType())
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("description", StringType())
    ]))),
    StructField("dt", LongType())
])

In [0]:
from pyspark.sql.functions import from_json, col

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), schema)
)

In [0]:
silver_df = parsed_df.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather_condition"),
    col("json_data.dt").alias("timestamp"),
    col("ingestion_time")
)

In [0]:
from pyspark.sql.functions import to_timestamp

silver_df = silver_df \
    .dropDuplicates() \
    .filter(col("city").isNotNull()) \
    .withColumn("event_time", to_timestamp(col("timestamp")))

In [0]:
silver_path = "abfss://processed@cloud1111.dfs.core.windows.net/silver/api/weather/"

silver_df.write \
    .format("delta") \
    .mode("append") \
    .save(silver_path)

In [0]:
df_check = spark.read.format("delta").load(silver_path)
display(df_check)

city,temperature,humidity,weather_condition,timestamp,ingestion_time,event_time
Bengaluru,30.28,52,scattered clouds,1778135354,2026-05-07T06:35:04.275616Z,2026-05-07T06:29:14Z
Bengaluru,32.65,41,scattered clouds,1778147761,2026-05-07T10:00:55.257605Z,2026-05-07T09:56:01Z
Delhi,31.05,37,clear sky,1778135536,2026-05-07T06:35:04.275616Z,2026-05-07T06:32:16Z
Delhi,31.05,40,clear sky,1778147940,2026-05-07T10:00:55.257605Z,2026-05-07T09:59:00Z
London,9.16,83,overcast clouds,1778135440,2026-05-07T06:35:04.275616Z,2026-05-07T06:30:40Z
London,12.96,65,broken clouds,1778147710,2026-05-07T10:00:55.257605Z,2026-05-07T09:55:10Z
New York,11.91,86,moderate rain,1778134972,2026-05-07T06:35:04.275616Z,2026-05-07T06:22:52Z
New York,10.28,86,overcast clouds,1778147661,2026-05-07T10:00:55.257605Z,2026-05-07T09:54:21Z
Paris,10.01,86,clear sky,1778135168,2026-05-07T06:35:04.275616Z,2026-05-07T06:26:08Z
Paris,14.76,61,broken clouds,1778147971,2026-05-07T10:00:55.257605Z,2026-05-07T09:59:31Z


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

In [0]:
silver_df = silver_df.dropDuplicates(["city", "event_time"])

In [0]:
from delta.tables import DeltaTable

silver_path = "abfss://processed@cloud1111.dfs.core.windows.net/silver/api/weather/"

# Check if table exists
if DeltaTable.isDeltaTable(spark, silver_path):

    silver_table = DeltaTable.forPath(spark, silver_path)

    silver_table.alias("target").merge(
        silver_df.alias("source"),
        "target.city = source.city AND target.event_time = source.event_time"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    # First time write
    silver_df.write.format("delta").mode("overwrite").save(silver_path)

In [0]:
silver_path = "abfss://processed@cloud1111.dfs.core.windows.net/silver/api/weather/"

silver_df = spark.read.format("delta").load(silver_path)

In [0]:
from pyspark.sql.functions import avg, max, min, to_date

gold_df = silver_df.withColumn("date", to_date("event_time")) \
    .groupBy("city", "date") \
    .agg(
        avg("temperature").alias("avg_temp"),
        max("temperature").alias("max_temp"),
        min("temperature").alias("min_temp"),
        avg("humidity").alias("avg_humidity")
    )

In [0]:
weather_dist_df = silver_df.groupBy("city", "weather_condition") \
    .count()

In [0]:
gold_path = "abfss://curated@cloud1111.dfs.core.windows.net/gold/api/weather/"

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("append") \
    .save(gold_path)

In [0]:
gold_df.write \
    .mode("overwrite") \
    .saveAsTable("gold_weather_daily")

In [0]:
%sql
SELECT * FROM gold_weather_daily
ORDER BY date DESC;

city,date,avg_temp,max_temp,min_temp,avg_humidity
Delhi,2026-05-07,31.05,31.05,31.05,38.0
Bengaluru,2026-05-07,31.416666666666668,32.65,30.28,46.666666666666664
New York,2026-05-07,11.104999999999999,11.91,10.28,85.66666666666667
Tokyo,2026-05-07,22.888333333333332,24.09,21.4,79.33333333333333
Paris,2026-05-07,12.446666666666667,15.13,10.01,72.83333333333333
London,2026-05-07,11.091666666666669,13.15,9.16,73.66666666666667
Tokyo,2026-05-04,22.81,22.81,22.81,44.0
London,2026-05-04,13.52,13.52,13.52,82.0
Bengaluru,2026-05-04,33.26,33.26,33.26,39.0
Paris,2026-05-04,15.94,15.94,15.94,87.0
